# Voyager

Simulating the Voyager 1 trajectory

In [2]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from numpy.linalg import norm
from scipy.integrate import solve_ivp
from scipy.optimize import fsolve
from scipy.interpolate import CubicSpline
import matplotlib.pyplot as plt
from lamberthub import vallado2013
import plotly.graph_objects as go
from tqdm import tqdm

from constants import DEG_TO_RAD, AU_TO_KM, MU_SUN, DAY_TO_SEC, R_SUN, HOUR_TO_SEC
from orbital import rv_from_coe
from plotting import plot_3d
from spacetime import greg_to_jd

In [ ]:
gaspra_r_HCI = np.array([7.635070746360724e7, -3.201967071615438e8, 1.17515936590249e7])  # km
gaspra_v_HCI = np.sqrt(MU_SUN/norm(gaspra_r_HCI)) * np.cross([0,0,1], gaspra_r_HCI) / norm(gaspra_r_HCI) # km/s

flybys = [
    OrbitalFlyby("Earth Departure", (1989, 10, 26, 0, 55, 0), 3),
    OrbitalFlyby("Venus Swingby", (1990, 2, 10, 5, 59, 0), 2),
    OrbitalFlyby("Earth Swingby #1", (1990, 12, 8, 20, 35, 0), 3),
    OrbitalFlyby("Gaspra Encounter", (1991, 10, 29, 22, 49, 0), None),
    OrbitalFlyby("Earth Swingby #2", (1992, 12, 8, 15, 9, 0), 3),
    OrbitalFlyby("Jupiter Arrival", (1995, 12, 7, 21, 53, 44), 5)
]
N_FLYBYS = len(flybys)

for flyby in flybys:
    if flyby.planet_num is None:
        flyby.add_rv_HCI(gaspra_r_HCI, gaspra_v_HCI)
    else:
        flyby.add_rv_HCI(*rv_HCI(flyby.planet_num, flyby.jd, eng))